<center><img src='https://raw.githubusercontent.com/Jangrae/img/master/satisfaction.png'/></center>

### **고객 만족도 예측을 통한 서비스 개선**
# **단계 1: 데이터 분석**

## **미션 설명**

- 이번 단계에서는 다음과 같이 5개의 미션을 수행합니다.

### 미션 1: 기본 탐색 및 전처리

- 분석 대상 데이터를 읽어와 기본적인 탐색을 수행합니다.
- 결측치 존재 여부를 확인하고 적절한 방법으로 처리합니다.
- Target 값을 숫자로 변경합니다.
- 분석에 불필요한 변수를 제거합니다.

### 미션 2: 가설 수립과 검증

- 탐색 과정을 통해 얻은 정보로 가설을 수립합니다.
- 데이터 시각화 분석을 통해 수립한 가설을 검증합니다.
- 모든 분석 과정에 대한 의견을 정리합니다.
- 정리된 의견은 팀 프로젝트 수행 시 발표 자료에 정리합니다.


### 미션 3: 머신러닝 모델링 #1

- 머신러닝 모델을 만들어 성능을 평가합니다.
- 변수 중요도를 확인하고 의견으로 정리합니다.

### 미션 4: 머신러닝 모델링 #2

- 데이터를 두 개로 분리합니다.
- 각각의 데이터에 대해 별도의 모델을 만들어 성능을 평가합니다.
- 각각의 모델이 중요하다고 판단한 변수를 확인하고 정리합니다.

### 미션 5: 데이터 분석

- 모델링 과정을 통해 얻은 통찰력으로 분석 대상 변수를 선택합니다.
- 선택한 변수에 대한 분석을 수행하고 결과를 의견으로 정리합니다.

## **※ 코드 셀은 충분히 추가해 사용합니다.**

## **1. 환경설정**

### (1) 구글 드라이브 연결 및 경로 설정

- 구글 드라이브에 **project01** 폴더를 만들고 배포한 파일을 업로드합니다.
- 다음 구문을 실행에 구글 코랩에서 사용 가능하게 연결합니다.

In [ ]:
# 구글 드라이브 연결 & 패스 지정
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    path = '/content/drive/MyDrive/project01/'
else:
    path = ''

### (2) 라이브러리 불러오기

- 이후에 사용할 라이브러리를 모두 불러옵니다.

In [ ]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import *
from sklearn.ensemble import RandomForestClassifier

import joblib
import warnings

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format='retina'

### (3) 데이터 불러오기

- 다음 파일을 읽어와 **data** 데이터프레임을 만듭니다.
    - path + 'survey.csv' → data

In [ ]:
# 파일 읽어오기
file = path + 'survey.csv'
data = pd.read_csv(file)

## **2. 미션 1: 기본 탐색 및 전처리**

### (1) 기본정보 확인

- 데이터 형태, 기초통계량 등을 확인합니다.

In [ ]:
# 크기(행, 열 개수) 확인
print(data.shape)

In [ ]:
# 상위 5개 행 확인
data.head()

In [ ]:
# 변수 정보 확인
data.info()

In [ ]:
# 기술통계량 확인
data.describe()

### (2) Target 변수 확인

- Target인 **Satisfaction** 변수에 대해 단변량 분석을 수행합니다.

In [ ]:
# 범줏값 확인
data['Satisfaction'].value_counts()

In [ ]:
# 범줏값 비율 확인
data['Satisfaction'].value_counts() / data.shape[0]

In [ ]:
# 시각화
sns.countplot(x='Satisfaction', data=data)
plt.show()

### (3) 결측치 확인

- 결측치 비율을 기준으로 내림차순 정렬해 확인합니다.

In [ ]:
# 결측치 비율 확인
data.isna().mean().sort_values(ascending=False)

### (4) 결측치 채우기

- 결측치가 있는 수치형 변수는 **중앙값**으로, 범주형 변수는 **최빈값**으로 채웁니다.
    - 수치형 변수: 'Age', 'Arrival Delay in Minutes'
    - 범주형 변수: 'Inflight wifi service', 'Seat comfort', 'Food and drink'
- 추천: 이후 확장성을 위해 범주형/수치형 변수를 자동 선택하고, **반복문**으로 결측치를 채워봅니다.

In [ ]:
# 수치형 변수 선택
num_cols = data.select_dtypes(include=['number']).columns

# 결측치 → 중앙값
for col in num_cols:
    data[col].fillna(data[col].median(), inplace=True)

In [ ]:
# 범주형 변수 선택
cat_cols = data.select_dtypes(include=['object', 'category']).columns

# 결측치 → 최빈값
for col in cat_cols:
    data[col].fillna(data[col].mode()[0], inplace=True)

In [ ]:
# 결측치 확인
data.isna().sum().sort_values(ascending=False)

### (5) Target 값 변경

- Target인 **Satisfaction** 변수  값을 1과 0으로 변경합니다.
    - 'Satisfied'--> 1,
    - 'Neutral or Dissatisfied' --> 0
- **map()** 메서드를 사용합니다.

In [ ]:
# 변경 전
print(data['Satisfaction'].value_counts())

In [ ]:
# 값 변경
data['Satisfaction'] = data['Satisfaction'].map({'Satisfied': 1, 'Neutral or Dissatisfied': 0})

In [ ]:
# 변경 후
print(data['Satisfaction'].value_counts())

### (6) 불필요한 변수 제거

- 다음 변수는 분석에 의미가 없으므로 제거합니다.
    - 'Unnamed: 0'
    - 'ID'

In [ ]:
# 변수 제거
drop_cols = ['Unnamed: 0', 'ID']
data.drop(columns=drop_cols, inplace=True)

# 확인
data.info()

## **3. 미션 2: 가설 수립과 검증**

### (1) 가설 수립

- 'ㅇㅇ이면 ㅇㅇ일 것이다' 라는 형식의 가설을 5개 이상 세워봅니다.
- 이후 분석 과정을 통해 수립한 가설을 검증합니다.
- 특히 이변량 분석을 통해 가설을 검증하도록 합니다.

In [ ]:
# 가설 1: 성별에 따라 만족 여부에 차이가 있을 것이다.
# 가설 2: 고객 유형에 따라 만족 여부에 차이가 있을 것이다.
# 가설 3: 여행 목적에 따라 만족 여부에 차이가 있을 것이다.
# 가설 4: 좌석 등급에 따라 만족 여부에 차이가 있을 것이다.
# 가설 5: 나이에 따라 만족 여부에 차이가 있을 것이다.

### (2) 단변량 분석

- 5개 이상의 변수에 대한 단변량 분석을 수행합니다.
- 각 분석에 대한 의견을 주석으로 정리합니다.

In [ ]:
# Gender
sns.countplot(x='Gender', data=data)
plt.show()

# 의견: 남녀 비율이 거의 비슷해 보임

In [ ]:
# Customer Type
sns.countplot(x='Customer Type', data=data)
plt.show()

# 의견: Loyal 고객 비율이 매우 높음

In [ ]:
# Type of Travel
sns.countplot(x='Type of Travel', data=data)
plt.show()

# 의견: 비즈니스 목적 여행 비율이 더 큼

In [ ]:
# Class
sns.countplot(x='Class', data=data)
plt.show()

# 의견: Eco, Eco Plus 등급 좌석 비율이 매우 낮음

### (3) 이변량 분석

- 5개 이상의 이변량 분석을 수행합니다.
- 각 분석에 대한 의견을 주석으로 정리합니다.

In [ ]:
# Gender → Satisfaction
table = pd.crosstab(data['Gender'], data['Satisfaction'], normalize='index')

table.plot(kind='bar', stacked=True)
plt.xticks(rotation=0)
plt.show()

# 의견: 성별에 따른 만족 여부에 차이가 거의 없어 보임

In [ ]:
# Customer Type → Satisfaction
table = pd.crosstab(data['Customer Type'], data['Satisfaction'], normalize='index')

table.plot(kind='bar', stacked=True)
plt.xticks(rotation=0)
plt.show()

# 의견: 고객 유형에 따라 만족 여부에 차이가 조금 있어 보임
#       특히, Disloyal인 경우 불만족 비율이 더 큼

In [ ]:
# Type of Travel → Satisfaction
table = pd.crosstab(data['Type of Travel'], data['Satisfaction'], normalize='index')

table.plot(kind='bar', stacked=True)
plt.xticks(rotation=0)
plt.show()

# 의견: 여행 목적에 따라 만족 여부에 큰 차이가 있어 보임
#       사적인 여행인 경우 불만족 비율이 현저하게 큼

In [ ]:
# Class → Satisfaction
table = pd.crosstab(data['Class'], data['Satisfaction'], normalize='index')

table.plot(kind='bar', stacked=True)
plt.xticks(rotation=0)
plt.show()

# 의견: 좌석 등급에 따라 만족 여부에 차이가 있어 보임
#       특히 Business 등급의 경우 차이가 두드러져 보임

In [ ]:
# Age → Satisfaction
sns.kdeplot(x='Age', hue='Satisfaction', data=data)
plt.show()

# 의견: 중년층(40세 ~ 50세)은 만족 비율이 더 큼

In [ ]:
# Flight Distance → Satisfaction
sns.kdeplot(x='Flight Distance', hue='Satisfaction', data=data)
plt.show()

# 의견: 비행 거리가 길 수록 불만족 비율이 더 큼

## **4. 미션 3: 머신러닝 모델링 #1**

- Tree 기반 알고리즘을 사용해야 **변수 중요도**를 확인할 수 있습니다.
- **RandomForest** 알고리즘을 사용해 머신러닝 모델을 만들고 성능을 평가합니다.
- **변수 중요도**를 확인해 추가 분석 대상 변수를 선정합니다.

### (1) 전처리

- 최소한 문자열 데이터가 없어야 하고, 결측치도 없도록 전처리해야 합니다.
- 결측치는 이전 단계에서 처리한 상태입니다.
- 범주형 변수에 대해 **LabelEncoder**를 사용해 인코딩합니다.
- 변수 중요도를 확인하기 위함이므로 가변수화를 하지 않습니다.

In [ ]:
# 변수 확인
data.info()

In [ ]:
# 라벨 인코딩
cat_cols = ['Gender', 'Customer Type', 'Type of Travel',  'Class']

for col in cat_cols:
    encoder = LabelEncoder()
    data[col] = encoder.fit_transform(data[col])

- info() 메서드로 변수 정보를 다시 확인합니다.
- **Dtype에 object 형식의 열이 있으면 이후 과정을 진행할 수 없습니다.**

In [ ]:
# 변수 확인
data.info()

- target 변수를 선정하고 x, y를 분리합니다.

In [ ]:
# x, y 분리
target = 'Satisfaction'
x = data.drop(target, axis=1)
y = data.loc[:, target]

- 학습용, 검증용 데이터를 7:3으로 분리합니다.

In [ ]:
# 학습용, 검증용 분리(7:3)
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=.3, stratify=y, random_state=1)

### (2) 모델링

- 머신러닝 모델링을 수행합니다.
- 검증 데이터로 예측하고 성능을 평가해 봅니다.

In [ ]:
# 선언하기
model = RandomForestClassifier(random_state=1)

# 학습하기
model.fit(x_train, y_train)

# 예측하기
y_pred = model.predict(x_val)

# 평가하기
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

### (3) 변수 중요도 확인

- 변수 중요도를 시각화해 확인합니다.

In [ ]:
# 변수 중요도 시각화
tmp = pd.DataFrame()
tmp['features'] = list(x)
tmp['importance'] = model.feature_importances_

# 정렬
tmp.sort_values(by='importance', inplace=True)

# 시각화
plt.figure(figsize=(5, 8))
plt.barh(y='features', width='importance', data=tmp)
plt.show()

### (4) 결과 정리

- 가장 중요하다고 확인된 변수 5개는 무엇인가요?


In [ ]:
# 확인
tmp.sort_values(by='importance', ascending=False).head()

In [ ]:
# 1. Inflight wifi service
# 2. Online boarding
# 3. Type of Travel
# 4. Ease of Online booking
# 5. Class

- 가장 중요하지 않다고 확인된 변수 5개는 무엇인가요?

In [ ]:
tmp.sort_values(by='importance', ascending=True).head()

In [ ]:
# 1. Gender
# 2. Food and drink
# 3. Gate location
# 4. Departure Delay in Minutes
# 5. Baggage handling

## **5. 미션 4: 머신러닝 모델링 #2**

- 35세를 기준으로 데이터를 분리해 각각에 대한 모델을 만들어봅니다.

### (1) 데이터 분리

- 더 세분화할 수 있지만, 35세를 기준으로 두 개의 데이터프레임을 만듭니다.
    - **data01**: Age <= 35
    - **data02**: Age > 35

In [ ]:
# 데이터 분리
data01 = data.loc[data['Age'] <= 35]
data02 = data.loc[data['Age'] > 35]

# 인덱스 초기화
data01.reset_index(drop=True, inplace=True)
data02.reset_index(drop=True, inplace=True)

# 데이터 크기(행, 열) 확인
print(data01.shape)
print(data02.shape)

### (2) 35세 이하 모델링

- **data01** 데이터프레임을 대상으로 모델링을 수행합니다.
- 우선 target 변수를 선정하고 x, y를 분리합니다.

In [ ]:
# x, y 분리
target = 'Satisfaction'
x = data01.drop(target, axis=1)
y = data01.loc[:, target]

- 학습용, 검증용 데이터를 7:3으로 분리합니다.

In [ ]:
# 학습용, 검증용 분리(7:3)
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=.3, stratify=y, random_state=1)

- 모델링을 수행합니다.

In [ ]:
# 선언하기
model = RandomForestClassifier(random_state=1)

# 학습하기
model.fit(x_train, y_train)

# 예측하기
y_pred = model.predict(x_val)

# 평가하기
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

- 변수 중요도를 확인합니다.

In [ ]:
# 변수 중요도 시각화
tmp = pd.DataFrame()
tmp['features'] = list(x)
tmp['importance'] = model.feature_importances_

# 정렬
tmp.sort_values(by='importance', inplace=True)

# 시각화
plt.figure(figsize=(5, 8))
plt.barh(y='features', width='importance', data=tmp)
plt.show()

- 중요하다고 확인된 변수 5개는 무엇인가요?


In [ ]:
# 확인
tmp.sort_values(by='importance', ascending=False).head()

In [ ]:
# 1. Online boarding
# 2. Inflight wifi service		
# 3. Ease of Online booking	
# 4. Type of Travel	
# 5. Class	

- 중요하지 않다고 확인된 변수 5개는 무엇인가요?

In [ ]:
tmp.sort_values(by='importance', ascending=True).head()

In [ ]:
# 1. Gender	
# 2. Gate location
# 3. Food and drink	
# 4. Departure/Arrival time convenient	
# 5. Customer Type	

### (3) 35세 초과 모델링

- **data02** 데이터프레임을 대상으로 모델링을 수행합니다.
- 우선 target 변수를 선정하고 x, y를 분리합니다.

In [ ]:
# x, y 분리
target = 'Satisfaction'
x = data02.drop(target, axis=1)
y = data02.loc[:, target]

- 학습용, 검증용 데이터를 7:3으로 분리합니다.

In [ ]:
# 학습용, 검증용 분리(7:3)
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=.3, stratify=y, random_state=1)

- 모델링을 수행합니다.

In [ ]:
# 선언하기
model = RandomForestClassifier(random_state=1)

# 학습하기
model.fit(x_train, y_train)

# 예측하기
y_pred = model.predict(x_val)

# 평가하기
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

- 변수 중요도를 확인합니다.

In [ ]:
# 변수 중요도 시각화
tmp = pd.DataFrame()
tmp['features'] = list(x)
tmp['importance'] = model.feature_importances_

# 정렬
tmp.sort_values(by='importance', inplace=True)

# 시각화
plt.figure(figsize=(5, 8))
plt.barh(y='features', width='importance', data=tmp)
plt.show()

- 가장 중요하다고 확인된 변수 5개는 무엇인가요?


In [ ]:
# 확인
tmp.sort_values(by='importance', ascending=False).head()

In [ ]:
# 1. Inflight wifi service
# 2. Type of Travel
# 3. Online boarding
# 4. Ease of Online booking
# 5. Leg room service

- 가장 중요하지 않다고 확인된 변수 5개는 무엇인가요?

In [ ]:
tmp.sort_values(by='importance', ascending=True).head()

In [ ]:
# 1. Gender
# 2. Gate location
# 3. Food and drink
# 4. Departure/Arrival time convenient
# 5. Customer Type

### (4) 의견 정리

- 위 모델링 과정과 변수 중요도 확인 과정 결과에 대한 의견을 정리합니다.

In [ ]:
# 좌석 등급에 따라 만족도에 영향을 끼치는 변수가 다름
# Business 석 모델링에서 중요 변수로 확인된 변수가 Eco 석 모델링에서는 중요하지 않는 변수로 확인됨
# 만족도를 높이기 위해 고객 유형에 따른 대응 방법이 달라야 할 것으로 보임

## **6. 미션 5: 데이터 분석**

- 위에서 확인된 변수 중요도를 기준으로 분석할 변수를 5개 선정하고 분석을 수행합니다.
- **data** 데이터프레임을 대상으로 분석합니다.
- Satisfaction 변수와의 **이변량 분석**에 중점을 두고 수행합니다.
- 각 분석 결과에 대한 의견을 주석으로 정리합니다.

In [ ]:
# Online boarding
table = pd.crosstab(data['Online boarding'], data['Satisfaction'], normalize='index')

table.plot(kind='bar', stacked=True)
plt.xticks(rotation=0)
plt.show()

# 의견: Online boarding 만족도가 4, 5 인 경우 만족 비율이 크게 높아짐
#       0인 경우도 만족 비율이 높음 --> 만족도 0에 대한 재정의 필요 할 듯

In [ ]:
# Inflight wifi service
table = pd.crosstab(data['Inflight wifi service'], data['Satisfaction'], normalize='index')

table.plot(kind='bar', stacked=True)
plt.xticks(rotation=0)
plt.show()

# 의견: Inflight wifi service 만족도가 5인 경우 만족 비율이 크게 높아짐
#       0인 경우도 만족 비율이 높음 --> 만족도 0에 대한 재정의 필요 할 듯

In [ ]:
# Ease of Online booking
table = pd.crosstab(data['Ease of Online booking'], data['Satisfaction'], normalize='index')

table.plot(kind='bar', stacked=True)
plt.xticks(rotation=0)
plt.show()

# 의견: Ease of Online booking 만족도가 4, 5인 경우 만족 비율이 크게 높아짐
#       0인 경우도 만족 비율이 높음 --> 만족도 0에 대한 재정의 필요 할 듯

In [ ]:
# Inflight entertainment
table = pd.crosstab(data['Inflight entertainment'], data['Satisfaction'], normalize='index')

table.plot(kind='bar', stacked=True)
plt.xticks(rotation=0)
plt.show()

# 의견: Inflight entertainment 만족도가 4, 5인 경우 만족 비율이 크게 높아짐
#       0인 경우도 만족 비율이 0 --> 다른 변수와 다른 상황

## **7. 저장하기**

- **to_csv()** 메서드를 사용해 이후 과정을 위해 data 데이터프레임을 **project01** 폴더에 저장합니다.
    - index=False 옵션 지정
    - data → path + 'data.csv'
    
- 저장된 파일을 필히 확인합니다.

In [ ]:
# 파일 저장
data.to_csv(path + 'data.csv', index=False)